In [1]:
import pandas as pd
from pathlib import Path

In [ ]:
import sys 
sys.path.append('..')
from module.dataload import DPN_data


In [ ]:
config_path = Path(r'experiments')

# choose between final and development config file
# config_filename = "bin_sel_dev.yml" # development
config_filename = "bin_sel_final.yml" # final

config_dict = ymlconfig.load_config(config_path / config_filename)
config = ymlconfig.dict_to_namespace(config_dict)
config_dict

{'experiment': {'summary': 'binary classification - feature  and model selection (final experiment)',
  'classification_type': 'binary',
  'stage': 'selection',
  'tag': 'final',
  'verbosity': 0,
  'random_seed': 42},
 'data': {'dataset_path': '../dataset/Sudoscan Working File with Stats.xlsx'},
 'feature_selection': {'cross_validation': {'k_splits': 4,
   'n_repeats': 10,
   'sort_by': 'auprc'},
  'vif_threshold': 5},
 'figures': {'summary_table_topk': 5}}

In [ ]:
D = DPN_data(config.data.dataset_path)
D.load(classification=config.experiment.classification_type)
D.df.tail(3)

------
     SEX  AGE SUBJ DM_DUR INSULIN  HBA1C  HPN PAOD DSLPDMIA  CKD  ...  \
191  NaN  NaN  NaN    NaN     NaN    NaN  NaN  NaN      NaN  NaN  ...   

    FEET_MEAN_ESC FEET_PCT_ASYM HAND_MEAN_ESC HAND_PCT_ASYM  NS    CAS  \
191           NaN           NaN           NaN           NaN NaN  TOTAL   

    Confirmed Probable Possible Any_DPN  
191     129.0    138.0    180.0   181.0  

[1 rows x 44 columns]


/home/toni_briza/dpn2026/module/../module/dataload.py:102: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('NR', 0, inplace=True)
/home/toni_briza/dpn2026/module/../module/dataload.py:103: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('NO F WAVE', 0, inplace=True)
/home/toni_briza/dpn2026/module/../module/dataload.py:104: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-i

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,...,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed_Binary_DPN
187,1,36.0,0,1.0,1.0,6.18,1,0,0,1,...,9.43,7.19,49.9,69.0,4.0,56.0,8.0,83.0,7.0,0
188,0,60.0,1,5.0,1.0,12.20,1,0,1,0,...,5.09,3.28,53.5,16.0,11.0,21.0,9.0,46.0,32.0,1
189,0,65.0,1,15.0,1.0,7.59,1,1,0,1,...,0.27,0.11,0.0,39.0,16.0,41.0,23.0,43.0,44.0,1


In [ ]:
df = pd.read_excel(config.data.dataset_path, skiprows=3, usecols="B:G, I:AT", names=D.col_names, na_values=['-'],
                           decimal=',')

(192, 44)

In [7]:
df

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,...,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed,Probable,Possible,Any_DPN
0,M,64.0,Y,7,Y,15.00,N,N,N,N,...,12.0,0.0,33.0,13.0,42.0,34,1.0,1.0,1.0,1.0
1,F,59.0,Y,1,N,5.60,Y,N,N,N,...,39.0,5.0,38.0,28.0,50.0,39,0.0,0.0,1.0,1.0
2,F,64.0,Y,11,Y,7.50,Y,N,N,N,...,65.0,14.0,79.0,1.0,50.0,33,1.0,1.0,1.0,1.0
3,F,53.0,Y,10,Y,7.60,Y,N,Y,N,...,43.0,10.0,49.0,5.0,57.0,33,1.0,1.0,1.0,1.0
4,M,57.0,N,5,Y,14.40,N,N,N,N,...,54.0,3.0,63.0,0.0,54.0,36,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,M,36.0,N,<1,Y,6.18,Y,N,N,Y,...,69.0,4.0,56.0,8.0,83.0,7,0.0,1.0,1.0,1.0
188,F,60.0,Y,5,Y,12.20,Y,N,Y,N,...,16.0,11.0,21.0,9.0,46.0,32,1.0,1.0,1.0,1.0
189,F,65.0,Y,15,Y,7.59,Y,Y,N,Y,...,39.0,16.0,41.0,23.0,43.0,44,1.0,1.0,1.0,1.0
190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df = df.dropna(how='all')
df.shape

(191, 44)

In [9]:
df = df[:-1]  # the sheet's last row is a totals row, not a subject record, so drop it
if df.shape != (190, len(D.col_names)):
    raise ValueError(f"Expected 190 rows x {len(D.col_names)} columns after cleanup, got {df.shape}")

In [10]:
df.shape

(190, 44)

In [11]:
df.replace('NR', 0, inplace=True)
df.replace('NO F WAVE', 0, inplace=True)
df.replace({'Y': 1, 'M': 1, 'N': 0, 'F': 0}, inplace=True)

/tmp/ipykernel_3356300/3490429237.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('NR', 0, inplace=True)
/tmp/ipykernel_3356300/3490429237.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('NO F WAVE', 0, inplace=True)
/tmp/ipykernel_3356300/3490429237.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_s

In [14]:
df[df.isna().any(axis=1)]

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,...,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed,Probable,Possible,Any_DPN
35,0,64.0,1,14,0.0,7.00,1,0,0,0,...,78.0,1.0,76.0,11.0,NaN,NaN,1.0,1.0,1.0,1.0
45,0,53.0,0,NaN,NaN,NaN,0,0,0,0,...,74.0,1.0,77.0,6.0,49.0,33.0,0.0,0.0,0.0,0.0
172,1,48.0,1,10,1.0,10.89,1,0,0,0,...,30.0,0.0,28.0,3.0,NaN,29.0,0.0,1.0,1.0,1.0


In [20]:
for i in {35, 45, 172}:
    s = df[df.isna().any(axis=1)].loc[i]
    print(i)
    print(s[s.isna()])

35
NS     NaN
CAS    NaN
Name: 35, dtype: object
172
NS    NaN
Name: 172, dtype: object
45
DM_DUR     NaN
INSULIN    NaN
HBA1C      NaN
Name: 45, dtype: object


In [21]:
search_text = 'NR'
mask = df.astype(str).apply(lambda col: col.str.contains(search_text, case=False)).any(axis=1)
df[mask]

,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,...,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,Confirmed,Probable,Possible,Any_DPN
0,M,64.0,Y,7,Y,15.00,N,N,N,N,...,12.0,0.0,33.0,13.0,42.0,34,1.0,1.0,1.0,1.0
2,F,64.0,Y,11,Y,7.50,Y,N,N,N,...,65.0,14.0,79.0,1.0,50.0,33,1.0,1.0,1.0,1.0
8,M,62.0,N,0,Y,14.36,N,N,N,N,...,72.0,2.0,61.0,19.0,56.0,31,1.0,0.0,1.0,1.0
9,F,44.0,Y,17,N,7.01,N,N,N,N,...,19.0,10.0,84.0,1.0,64.0,18,1.0,1.0,1.0,1.0
15,M,60.0,Y,7,Y,9.40,Y,N,N,Y,...,36.0,20.0,9.0,10.0,48.0,40,1.0,1.0,1.0,1.0
16,F,47.0,Y,24,Y,7.40,Y,N,N,N,...,29.0,6.0,70.0,12.0,62.0,26,1.0,1.0,1.0,1.0
24,F,50.0,Y,30,Y,12.05,N,N,Y,Y,...,33.0,0.0,71.0,16.0,59.0,22,1.0,1.0,1.0,1.0
26,F,48.0,Y,12,N,10.00,N,N,Y,N,...,51.0,14.0,65.0,10.0,64.0,27,1.0,1.0,1.0,1.0
31,F,71.0,Y,30,N,7.50,Y,N,Y,N,...,39.0,2.0,42.0,38.0,37.0,38,1.0,1.0,1.0,1.0
36,M,45.0,Y,13,Y,7.69,N,N,N,Y,...,8.0,11.0,18.0,5.0,62.0,24,1.0,1.0,1.0,1.0
